<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Bank ClickStream - Outcome Prediction Model with First Order Markov Chains
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:20px;font-family:Arial"><b>Introduction</b></p>

<p style="font-size:16px;font-family:Arial">
User interactions with banking services across web, mobile, chatbot, phone, and branch visits leave behind event trails. Most activity is routine (bill pay, viewing statements), but some journeys lead to valuable outcomes like product applications. Identifying these high-propensity journeys in real-time enables targeted campaigns and personalized offers.</p>
<p style="font-size:18px;font-family:Arial"><b>What Does It Do?</b></p>
<li style="font-size:16px;font-family:Arial">Event Sequence Classification — Given a user's session events (e.g., PayBills → CheckCreditScore → CompareCreditCards → CheckCreditScore), predict which business outcome they are heading toward (e.g., Apply Credit Card — 98.5%). This is a multi-class/multi-label classification problem.</li>

<li style="font-size:16px;font-family:Arial">Next Event(s) Prediction — Given an in-progress session (e.g., …→ TransferFunds → ViewChecking → ViewSavings →), generate the likely sequence of future events the user will take. This is a generative sequence modeling problem.</li>    
</p>

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>1. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. </p>

In [ ]:
from getpass import getpass
from teradataml import *
import os

# Suppress warnings
import warnings

warnings.filterwarnings('ignore')
display.suppress_vantage_runtime_warnings = True

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=2._Bank_ClickStream_-_Outcome_Prediction_Model_with_First_Order_Markov_Chains.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'> <b> 2. Data exploring  </b></p>

<p style="font-size:16px;font-family:Arial">
Fetch Data from the Teradata Database.</p>


In [ ]:
#training_table = "banking_data_train"
df = DataFrame(in_schema("DEMO_Bank","Session_Events"))
df

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'> <b> 3. Train-Test Split</b></p>
<p style = 'font-size:16px;font-family:Arial'>The Dataset is split already into Train and Test Datasets. We will fetch these directly from the database.</p>

In [ ]:
df_train=DataFrame(in_schema("DEMO_Bank","Session_Events_Train"))
df_test= DataFrame(in_schema("DEMO_Bank","Session_Events_Test"))

In [ ]:
df_train.shape

In [ ]:
df_train

In [ ]:
##
## CHOOSE TARGET FOR CLASSIFICATION MODEL
##

#classification_target = 'Apply Credit Card'
#classification_target = 'Apply Auto Loan'
#classification_target = 'Apply Checking Account'
classification_target = 'ApplyMortgage'
# classification_target = 'Apply Personal Loan'
#classification_target = 'Apply Savings Account'
#classification_target = 'Apply Teen Checking'

In [ ]:
training_table = "DEMO_Bank.Session_Events_Train"

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'> <b> 3. Model Training</b></p>
<p style = 'font-size:16px;font-family:Arial'>Build 2xEvent Transition Probabilities on outcome and non-outcome sessions.  Sessions containing the target event (e.g., ApplyMortgage) are labeled as outcome (1); all others as non-outcome (0)</p>  

In [ ]:
execute_sql("""
    create volatile table outcome_sessions
    as
    (
    	select distinct UserId, SessionId
    	from {0}
    	where Event like '{1}'
    ) with data
    primary index (UserId, SessionId)
    on commit preserve rows
""".format(training_table, classification_target)
)

In [ ]:
execute_sql("""
    create volatile table non_outcome_sessions
    as
    (
       select distinct UserId, SessionId
       from {0}
       where (UserId, SessionId) not in (select * from outcome_sessions)
    ) with data
    primary index(UserId, SessionId)
    on commit preserve rows
""".format(training_table)
)

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial'><b>3.1 Model Tables Creation</b></p>

<p style = 'font-size:16px;font-family:Arial'>We get all the unique events occuring and than create the model table with 150 events × 2 classes = 300 rows. Count event occurrences per class with Laplacian smoothing.</p>

In [ ]:
execute_sql("""
    create volatile table all_unique_events as
    (
    	select distinct Event as Event from {0}
    )
    with data
    primary index(Event)
    on commit preserve rows
""".format(training_table)
)

In [ ]:
df = DataFrame("all_unique_events")
df.shape

In [ ]:
try:
    db_drop_table("first_order_markov_model")
except:
    False
    
execute_sql("""
    create volatile table first_order_markov_model as
    (
    	select cast(1 as integer) as outcome, 
               a.Event as Event_1, 
               b.Event as Event_2, 
               cast(0 as float) as counts, 
               cast(0 as float) as probability
    	from all_unique_events a,
             all_unique_events b
    	union all
    	select cast(0 as integer) as outcome,
               a.Event as Event_1, 
               b.Event as Event_2, 
               cast(0 as float) as counts, 
               cast(0 as float) as probability
    	from all_unique_events a,
             all_unique_events b
    )
    with data
    primary index(Event_1, Event_2, outcome)
    on commit preserve rows
""")

In [ ]:
df = DataFrame("first_order_markov_model")
df.shape

In [ ]:
df

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial'><b>3.2 Update Markov Model Table with Transition Event counts and Normalize with Laplacians</b></p>
<p style = 'font-size:16px;font-family:Arial'>Using the nPath function we check the User and Sessions for occurences of Event_1 and Event_2 and get the counts for the sames inthe model table.</p>

In [ ]:
execute_sql("""
    update first_order_markov_model
    from (
        select Event_1, 
               Event_2,
               count(distinct UserId || '-' || cast(SessionId as varchar(10))) as counts
        from 
            (
                select UserId, SessionId, Event_1, Event_2
            	from npath(
            		on (select * from {0})
            			partition by UserId, SessionId
            			order by "Event_TS"
            		using
            			mode(overlapping)
            			pattern('A.B')
            			Symbols(
            			  TRUE as A,
                          TRUE as B
            			)
            			Result(
                          first(UserId of A) as UserId,
                          first(SessionId of A) as SessionId,
            			  first(Event of A) as Event_1,
                          first(Event of B) as Event_2
            			)
            			Filter(
            			  FIRST("Event_TS" + interval '1' day OF ANY (A)) >
            			  FIRST("Event_TS" of ANY(B))
            			)
            	) as dt2
            ) as x
        where (UserId, SessionId) in (select UserId, SessionId from outcome_sessions)
          and (Event_1 not like '{1}' or Event_2 not like '{2}')
        group by 1,2
    ) x
    set counts = x.counts
    where first_order_markov_model.outcome = 1
      and first_order_markov_model.Event_1 = x.Event_1
      and first_order_markov_model.Event_2 = x.Event_2
""".format(training_table,classification_target,classification_target)
)

In [ ]:
execute_sql("""
    update first_order_markov_model
    from (
        select Event_1, 
               Event_2,
               count(distinct UserId || '-' || cast(SessionId as varchar(10))) as counts
        from 
            (
                select UserId, SessionId, Event_1, Event_2
            	from npath(
            		on (select * from {0})
            			partition by UserId, SessionId
            			order by "Event_TS"
            		using
            			mode(overlapping)
            			pattern('A.B')
            			Symbols(
            			  TRUE as A,
                          TRUE as B
            			)
            			Result(
                          first(UserId of A) as UserId,
                          first(SessionId of A) as SessionId,
            			  first(Event of A) as Event_1,
                          first(Event of B) as Event_2
            			)
            			Filter(
            			  FIRST("Event_TS" + interval '1' day OF ANY (A)) >
            			  FIRST("Event_TS" of ANY(B))
            			)
            	) as dt2
            ) as x
        where (UserId, SessionId) in (select UserId, SessionId from non_outcome_sessions)
          and (Event_1 not like '{1}' or Event_2 not like '{2}')
        group by 1,2
    ) x
    set counts = x.counts
    where first_order_markov_model.outcome = 0
      and first_order_markov_model.Event_1 = x.Event_1
      and first_order_markov_model.Event_2 = x.Event_2
""".format(training_table,classification_target,classification_target)
)

In [ ]:
df = DataFrame("first_order_markov_model")
df[df['counts'] == 0]

In [ ]:
## Add laplacian for the zero counts
execute_sql("""
    update first_order_markov_model set counts = counts + 1
""")

<p style = 'font-size:16px;font-family:Arial'> We calculate the propbability of occurence of the Events in both the outcomes for the target defined</p>

In [ ]:
## Fix the probability column for both outcome and non-outcome across all events and outcome/no-outcome
execute_sql("""
    update first_order_markov_model 
    from ( select sum(counts) as total from first_order_markov_model where outcome = 1) as x
    set probability = counts/x.total
    where outcome = 1
;
""")

In [ ]:
execute_sql("""
    update first_order_markov_model 
    from ( select sum(counts) as total from first_order_markov_model where outcome = 0) as x
    set probability = counts/x.total
    where outcome = 0
;
""")

In [ ]:
df = DataFrame("first_order_markov_model").to_pandas().reset_index()

In [ ]:
df[df['outcome'] == 1].sort_values(by='probability',ascending=False)

In [ ]:
df[df['outcome'] == 0].sort_values(by='probability',ascending=False)

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>4. Prepare Test Set with Ground Truth Separation and run Prediction Logic</b></p>
<p style = 'font-size:16px;font-family:Arial'>We follow the same process for the test dataset</p>

In [ ]:
#test_holdout_table = "banking_data_test"
test_holdout_table = "DEMO_Bank.Session_Events_Test"

In [ ]:
df = DataFrame(in_schema("DEMO_Bank","Session_Events_Test"))

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
execute_sql("""
    create volatile table test_outcome_sessions
    as
    (
    	select distinct UserId, SessionId
    	from {0}
    	where Event like '{1}'
    ) with data
    primary index(UserId, SessionId)
    on commit preserve rows
""".format(test_holdout_table, classification_target)
)

In [ ]:
execute_sql("""
    create volatile table test_non_outcome_sessions
    as
    (
       select distinct UserId, SessionId
       from {0}
       where (UserId, SessionId) not in (select * from test_outcome_sessions)
    ) with data
    primary index(UserId, SessionId)
    on commit preserve rows
""".format(test_holdout_table)
)

In [ ]:
execute_sql("""
     create volatile table test_scored_table
     as
     (
       select UserId, SessionId, transitions_per_session, log_odds, 1 / (1 + EXP(-log_odds)) AS score -- doing a sigmoid here
       from(
            select UserId, SessionId, count(*) as transitions_per_session, sum(log(o.probability/n.probability)) as log_odds
            from (
                    select UserId, SessionId, Event_1, Event_2
                	from npath(
                		on (select * from {0})
                			partition by UserId, SessionId
                			order by "Event_TS"
                		using
                			mode(overlapping)
                			pattern('A.B')
                			Symbols(
                			  TRUE as A,
                              TRUE as B
                			)
                			Result(
                              first(UserId of A) as UserId,
                              first(SessionId of A) as SessionId,
                			  first(Event of A) as Event_1,
                              first(Event of B) as Event_2
                			)
                			Filter(
                			  FIRST("Event_TS" + interval '1' day OF ANY (A)) >
                			  FIRST("Event_TS" of ANY(B))
                			)
                        )
                 ) a,
                 first_order_markov_model o,
                 first_order_markov_model n
            where o.outcome = 1 and 
                  n.outcome = 0 and     
                  a.Event_1 = o.Event_1 and
                  a.Event_2 = o.Event_2 and
                  a.Event_1 = n.Event_1 and
                  a.Event_2 = n.Event_2                  
            group by 1,2
        ) x
     ) with data
     primary index(UserId, SessionId)
     on commit preserve rows
""".format(test_holdout_table)
)

In [ ]:
df = DataFrame("test_scored_table")

In [ ]:
df[df['log_odds'] > 0]

In [ ]:
df[df['log_odds'] < 0]

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>5. Evaluation wrt Ground Truth/Precision Recall Metrics</b></p>
 <p style = 'font-size:16px;font-family:Arial'>We evaluate the test data, calculate the probabilities and than get the model accuracy and plot the confusion matrix for the model</p>

In [ ]:
execute_sql("""
    create volatile table test_evaluation_table
    as
    (
        select a.UserId, a.SessionId, a.transitions_per_session, a.score, b.outcome
        from
            test_scored_table a
            inner join
            (
              select 1 as outcome, UserId, SessionId 
                   from test_outcome_sessions
              union all
              select 0 as outcome, UserId, SessionId 
                   from test_non_outcome_sessions
            ) b on (a.UserId = b.UserId and a.SessionId = b.SessionId)
    ) with data
    primary index(UserId,SessionId)
    on commit preserve rows
""")

In [ ]:
df = DataFrame("test_evaluation_table").to_pandas().reset_index()

In [ ]:
df.shape

In [ ]:
df[df['outcome'] == 0]

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dfp = df[df["transitions_per_session"] > 1]

In [ ]:
thresholds = [0.1, 0.5, 0.7, 0.9]
pythonfig, axes = plt.subplots(1, len(thresholds), figsize=(4*len(thresholds), 4))

for ax, threshold in zip(axes, thresholds):
    predicted = (dfp['score'] >= threshold).astype(int)
    cm = confusion_matrix(dfp['outcome'], predicted)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'Threshold = {threshold}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    """Create confusion matrix without sklearn"""
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    return np.array([[tn, fp], [fn, tp]])



for threshold in thresholds:
    # Convert probabilities to binary predictions
    predicted = (df['score'] >= threshold).astype(int)
    
    # Generate confusion matrix
    cm = confusion_matrix_manual(df['outcome'], predicted)
    
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    print(f"\n--- Threshold: {threshold} ---")
    print(f"Confusion Matrix:")
    print(f"              Predicted 0  Predicted 1")
    print(f"Actual 0      {tn:>10}  {fp:>10}")
    print(f"Actual 1      {fn:>10}  {tp:>10}")
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")

In [ ]:
df.groupby('outcome')['score'].describe()

In [ ]:
df[df['outcome'] == 1]['score'].mean()  # Should be higher

In [ ]:
df[df['outcome'] == 0]['score'].mean()  # Should be lower

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>6. Multi-Class Classification</b></p>

<p style = 'font-size:16px;font-family:Arial'>Instead of asking "Will they apply for a mortgage?" (binary), we now ask **"Which product will they apply for?"** (multiclass).</p>

<p style = 'font-size:16px;font-family:Arial'>**Approach:** Train one binary model per Apply target class. For each test session, score it against all class models.</p>
<p style = 'font-size:16px;font-family:Arial'>The predicted class is the one with the highest log-likelihood ratio (log-odds). Sessions with all negative log-odds are classified as NoApplication.</p>

<p style = 'font-size:16px;font-family:Arial'>**Target classes discovered automatically** from the training data.</p>

In [ ]:
# Discover all Apply* events from training data
apply_events_df = DataFrame.from_query("""
    SELECT DISTINCT Event FROM {0} WHERE Event LIKE 'Apply%'
""".format(training_table))
apply_targets = apply_events_df.to_pandas()['Event'].tolist()
print(f"Discovered {len(apply_targets)} target classes: {apply_targets}")

In [ ]:
# Train one First-Order Markov Chain model per target class

for target in apply_targets:
    model_table = f"mk_model_{target}"
    print(f"\nTraining Markov Chain for: {target}")
    
    try:
        db_drop_table(model_table)
    except:
        pass
    
    # Outcome / non-outcome sessions for this target
    execute_sql(f"""
        CREATE VOLATILE TABLE mc_outcome_sess AS (
            SELECT DISTINCT UserId, SessionId FROM {training_table}
            WHERE Event = '{target}'
        ) WITH DATA PRIMARY INDEX(UserId, SessionId) ON COMMIT PRESERVE ROWS
    """)
    execute_sql(f"""
        CREATE VOLATILE TABLE mc_non_outcome_sess AS (
            SELECT DISTINCT UserId, SessionId FROM {training_table}
            WHERE (UserId, SessionId) NOT IN (SELECT * FROM mc_outcome_sess)
        ) WITH DATA PRIMARY INDEX(UserId, SessionId) ON COMMIT PRESERVE ROWS
    """)
    
    # Create transition model table
    execute_sql(f"""
        CREATE TABLE {model_table} , STORAGE = TD_OFSSTORAGE AS (
            SELECT CAST(1 AS INTEGER) AS outcome, a.Event AS Event_1, b.Event AS Event_2,
                   CAST(0 AS FLOAT) AS counts, CAST(0 AS FLOAT) AS probability
            FROM all_unique_events a, all_unique_events b
            UNION ALL
            SELECT CAST(0 AS INTEGER) AS outcome, a.Event AS Event_1, b.Event AS Event_2,
                   CAST(0 AS FLOAT) AS counts, CAST(0 AS FLOAT) AS probability
            FROM all_unique_events a, all_unique_events b
        ) WITH DATA PRIMARY INDEX(Event_1, Event_2, outcome)
    """)
    
    # Count transitions in outcome sessions
    execute_sql(f"""
        UPDATE {model_table}
        FROM (
            SELECT Event_1, Event_2,
                   COUNT(DISTINCT UserId || '-' || CAST(SessionId AS VARCHAR(10))) AS counts
            FROM (
                SELECT UserId, SessionId, Event_1, Event_2
                FROM npath(
                    ON (SELECT * FROM {training_table})
                    PARTITION BY UserId, SessionId ORDER BY \"Event_TS\"
                    USING mode(overlapping) pattern('A.B')
                    Symbols(TRUE AS A, TRUE AS B)
                    Result(first(UserId OF A) AS UserId, first(SessionId OF A) AS SessionId,
                           first(Event OF A) AS Event_1, first(Event OF B) AS Event_2)
                    Filter(FIRST(\"Event_TS\" + INTERVAL '1' DAY OF ANY(A)) > FIRST(\"Event_TS\" OF ANY(B)))
                ) AS dt2
            ) AS x
            WHERE (UserId, SessionId) IN (SELECT UserId, SessionId FROM mc_outcome_sess)
              AND Event_1 NOT LIKE 'Apply%' AND Event_2 NOT LIKE 'Apply%'
            GROUP BY 1, 2
        ) x
        SET counts = x.counts
        WHERE {model_table}.outcome = 1
          AND {model_table}.Event_1 = x.Event_1 AND {model_table}.Event_2 = x.Event_2
    """)
    
    # Count transitions in non-outcome sessions
    execute_sql(f"""
        UPDATE {model_table}
        FROM (
            SELECT Event_1, Event_2,
                   COUNT(DISTINCT UserId || '-' || CAST(SessionId AS VARCHAR(10))) AS counts
            FROM (
                SELECT UserId, SessionId, Event_1, Event_2
                FROM npath(
                    ON (SELECT * FROM {training_table})
                    PARTITION BY UserId, SessionId ORDER BY \"Event_TS\"
                    USING mode(overlapping) pattern('A.B')
                    Symbols(TRUE AS A, TRUE AS B)
                    Result(first(UserId OF A) AS UserId, first(SessionId OF A) AS SessionId,
                           first(Event OF A) AS Event_1, first(Event OF B) AS Event_2)
                    Filter(FIRST(\"Event_TS\" + INTERVAL '1' DAY OF ANY(A)) > FIRST(\"Event_TS\" OF ANY(B)))
                ) AS dt2
            ) AS x
            WHERE (UserId, SessionId) IN (SELECT UserId, SessionId FROM mc_non_outcome_sess)
              AND Event_1 NOT LIKE 'Apply%' AND Event_2 NOT LIKE 'Apply%'
            GROUP BY 1, 2
        ) x
        SET counts = x.counts
        WHERE {model_table}.outcome = 0
          AND {model_table}.Event_1 = x.Event_1 AND {model_table}.Event_2 = x.Event_2
    """)
    
    # Laplacian smoothing + normalize
    execute_sql(f"UPDATE {model_table} SET counts = counts + 1")
    execute_sql(f"""
        UPDATE {model_table}
        FROM (SELECT SUM(counts) AS total FROM {model_table} WHERE outcome = 1) AS x
        SET probability = counts / x.total WHERE outcome = 1
    """)
    execute_sql(f"""
        UPDATE {model_table}
        FROM (SELECT SUM(counts) AS total FROM {model_table} WHERE outcome = 0) AS x
        SET probability = counts / x.total WHERE outcome = 0
    """)
    
    try:
        execute_sql("DROP TABLE mc_outcome_sess")
        execute_sql("DROP TABLE mc_non_outcome_sess")
    except:
        pass
    
    print(f"  {model_table}: {DataFrame(model_table).shape[0]} rows")

print(f"\nDone. Trained {len(apply_targets)} Markov Chain models.")

In [ ]:
# Score all test sessions against each Markov class model

npath_sql = f"""
    SELECT UserId, SessionId, Event_1, Event_2
    FROM npath(
        ON (SELECT * FROM {test_holdout_table})
        PARTITION BY UserId, SessionId ORDER BY \"Event_TS\"
        USING mode(overlapping) pattern('A.B')
        Symbols(TRUE AS A, TRUE AS B)
        Result(first(UserId OF A) AS UserId, first(SessionId OF A) AS SessionId,
               first(Event OF A) AS Event_1, first(Event OF B) AS Event_2)
        Filter(FIRST(\"Event_TS\" + INTERVAL '1' DAY OF ANY(A)) > FIRST(\"Event_TS\" OF ANY(B)))
    )
"""

score_parts = []
for target in apply_targets:
    model_table = f"mk_model_{target}"
    score_parts.append(f"""
        SELECT UserId, SessionId, CAST('{target}' AS VARCHAR(50)) AS target_class,
               SUM(LOG(o.probability / n.probability)) AS log_odds,
               1.0 / (1.0 + EXP(-SUM(LOG(o.probability / n.probability)))) AS score
        FROM ({npath_sql}) a
        JOIN {model_table} o ON o.outcome = 1 AND a.Event_1 = o.Event_1 AND a.Event_2 = o.Event_2
        JOIN {model_table} n ON n.outcome = 0 AND a.Event_1 = n.Event_1 AND a.Event_2 = n.Event_2
        GROUP BY UserId, SessionId
    """)

union_sql = " UNION ALL ".join(score_parts)

try:
    execute_sql("DROP TABLE mc_nb_all_scores")
except:
    pass

execute_sql(f"""
    CREATE TABLE mc_nb_all_scores, STORAGE = TD_OFSSTORAGE AS (
        {union_sql}
    ) WITH DATA PRIMARY INDEX(UserId, SessionId, target_class)
""")

print(f"Scored {DataFrame('mc_nb_all_scores').shape[0]} (session, class) pairs")

In [ ]:
# Classify: each session gets the class with the highest log-odds
# Sessions where best log-odds < 0 => NoApplication

try:
    execute_sql("DROP TABLE mc_predictions")
except:
    pass

execute_sql("""
    CREATE TABLE mc_predictions, STORAGE = TD_OFSSTORAGE AS (
        SELECT UserId, SessionId, target_class AS predicted_class, log_odds, score
        FROM mc_nb_all_scores
        QUALIFY ROW_NUMBER() OVER (PARTITION BY UserId, SessionId ORDER BY log_odds DESC) = 1
    ) WITH DATA PRIMARY INDEX(UserId, SessionId)
""")

# Override: if best log_odds < 0, classify as NoApplication
execute_sql("""
    UPDATE mc_predictions SET predicted_class = 'NoApplication' WHERE log_odds < 0
""")

print("Predictions:")
pred_df = DataFrame('mc_predictions').to_pandas()
print(pred_df['predicted_class'].value_counts())

In [ ]:
# Build ground truth: actual class per test session
try:
    execute_sql("DROP TABLE mc_ground_truth")
except:
    pass

execute_sql(f"""
    CREATE TABLE mc_ground_truth, STORAGE = TD_OFSSTORAGE AS (
        SELECT UserId, SessionId,
               CAST(COALESCE(MAX(CASE WHEN Event LIKE 'Apply%' THEN Event END), 'NoApplication') AS VARCHAR(50)) AS true_class
        FROM {test_holdout_table}
        GROUP BY UserId, SessionId
    ) WITH DATA PRIMARY INDEX(UserId, SessionId)
""")

gt_df = DataFrame('mc_ground_truth').to_pandas()
print("Ground truth distribution:")
print(gt_df['true_class'].value_counts())

In [ ]:
# Join predictions with ground truth and evaluate
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

eval_df = DataFrame.from_query("""
    SELECT g.UserId, g.SessionId, g.true_class, 
           COALESCE(p.predicted_class, 'NoApplication') AS predicted_class
    FROM mc_ground_truth g
    LEFT JOIN mc_predictions p ON g.UserId = p.UserId AND g.SessionId = p.SessionId
""").to_pandas()

print(f"\nTotal test sessions evaluated: {len(eval_df)}")
print(f"\nOverall Accuracy: {accuracy_score(eval_df['true_class'], eval_df['predicted_class']):.4f}")
print(f"\nClassification Report:")
print(classification_report(eval_df['true_class'], eval_df['predicted_class'], zero_division=0))

In [ ]:
# Multiclass confusion matrix heatmap
import matplotlib.pyplot as plt
import seaborn as sns

labels = sorted(eval_df['true_class'].unique())
cm = confusion_matrix(eval_df['true_class'], eval_df['predicted_class'], labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision, recall, F1 summary table
from sklearn.metrics import precision_recall_fscore_support

p, r, f1, sup = precision_recall_fscore_support(
    eval_df['true_class'], eval_df['predicted_class'], labels=labels, zero_division=0
)
summary = pd.DataFrame({
    'Class': labels, 'Precision': p, 'Recall': r, 'F1': f1, 'Support': sup
}).set_index('Class')
print(summary.to_string())
print(f"\nWeighted F1: {(summary['F1'] * summary['Support']).sum() / summary['Support'].sum():.4f}")

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>7. Cleanup </b></p>
<p style = 'font-size:16px;font-family:Arial'>Since we are going to need the model tables in the next notebook currently we are not executing the cleanup process but after completion of the last part we can run the cleanup to drop all the model tables.</p>

In [ ]:
# Cleanup multiclass tables (optional)
# for target in apply_targets:
#     try: db_drop_table(f"mk_model_{target}")
#     except: pass
# for tbl in ['mc_nb_all_scores', 'mc_predictions', 'mc_ground_truth']:
#     try: db_drop_table(tbl)
#     except: pass

In [ ]:
remove_context()

<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>